# FIDE Chess Ratings: Player Class Predictor 🏆

## What is this notebook about?
FIDE is the international chess federation that assigns a **rating** to every registered chess player worldwide.

Each player also has a **K-factor** — a number (10, 20, or 40) that controls how fast their rating changes after a game:
- **K = 40** → New/lower-rated players. Rating moves quickly.
- **K = 20** → Mid-level players. Rating moves at a moderate pace.
- **K = 10** → Top players (rating ≥ 2400 & played 30+ games). Very stable.

### 🎯 Our Goal
**Predict the K-factor class of a chess player** using features like their rating, number of games played, birth year, federation, and sex.

This is a **multi-class classification** problem with 3 classes: {10, 20, 40}.

### 📋 Dataset
- **201,015 FIDE-registered players** worldwide
- Source: FIDE official ratings list


## Step 1: Import Libraries
We start by importing all the Python libraries we need.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, ConfusionMatrixDisplay)
import warnings
warnings.filterwarnings('ignore')

# Make plots look nice
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')
print("✅ All libraries imported successfully!")


## Step 2: Load the Dataset
We load the CSV file into a **DataFrame** — think of it as a table in Python.

In [ ]:
df = pd.read_csv('/kaggle/input/fide-chess-ratings/03_FIDE_Chess_Ratings.csv')
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()


## Step 3: Exploratory Data Analysis (EDA)
Before building models, we need to **understand our data**. Let's explore it!

In [ ]:
# Basic info
print("Column names and data types:")
print(df.dtypes)
print()
print("Missing values per column:")
print(df.isnull().sum())


In [ ]:
# Statistical summary of numeric columns
df[['rating', 'games', 'bday']].describe().round(2)


In [ ]:
# Target variable distribution
print("K-factor class counts:")
print(df['k'].value_counts())

plt.figure(figsize=(7, 4))
df['k'].value_counts().sort_index().plot(kind='bar', color=['#2196F3','#4CAF50','#FF5722'],
                                          edgecolor='black')
plt.title('Distribution of K-factor (Target Variable)', fontsize=14, fontweight='bold')
plt.xlabel('K-factor')
plt.ylabel('Number of Players')
plt.xticks(rotation=0)
for i, v in enumerate(df['k'].value_counts().sort_index().values):
    plt.text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Rating distribution by K-factor
plt.figure(figsize=(10, 5))
for k_val, color in zip([10, 20, 40], ['red', 'green', 'blue']):
    subset = df[df['k'] == k_val]['rating']
    plt.hist(subset, bins=50, alpha=0.5, label=f'K={k_val}', color=color)
plt.title('Rating Distribution by K-factor', fontsize=14, fontweight='bold')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Gender distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Overall gender
df['sex'].value_counts().plot(kind='pie', ax=axes[0], autopct='%1.1f%%',
                               colors=['#2196F3','#E91E63'], startangle=90)
axes[0].set_title('Gender Distribution', fontweight='bold')
axes[0].set_ylabel('')

# Rating by gender (box plot)
df.boxplot(column='rating', by='sex', ax=axes[1], 
           notch=False, patch_artist=True)
axes[1].set_title('Rating by Gender')
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('Rating')
plt.suptitle('')
plt.tight_layout()
plt.show()


In [ ]:
# Top 10 federations by player count
top_feds = df['fed'].value_counts().head(10)
plt.figure(figsize=(10, 4))
top_feds.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Top 10 Federations by Player Count', fontsize=14, fontweight='bold')
plt.xlabel('Federation')
plt.ylabel('Number of Players')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Average rating by K-factor
avg_rating = df.groupby('k')['rating'].mean().round(1)
print("Average rating per K-factor class:")
print(avg_rating)

# Correlation heatmap
plt.figure(figsize=(7, 5))
corr = df[['rating', 'games', 'bday', 'k']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', 
            linewidths=0.5, annot_kws={'size': 12})
plt.title('Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## Step 4: Feature Engineering
**Feature engineering** means creating new or transformed columns that help the model learn better.

We'll:
1. Fill missing values
2. Encode categorical columns (convert text → numbers)
3. Create a new `age` feature from birth year
4. Drop columns we don't need


In [ ]:
# Make a copy so we don't modify the original
df_model = df.copy()

# 1. Create 'has_title' feature — does the player hold any FIDE title?
df_model['has_title'] = (~df_model['title'].isnull()).astype(int)

# 2. Approximate age from birth year (ratings are typically from 2024)
df_model['age'] = 2024 - df_model['bday']

# 3. Encode 'sex': M=1, F=0
df_model['sex_encoded'] = (df_model['sex'] == 'M').astype(int)

# 4. Encode 'fed' (federation) — convert country codes to numbers
le = LabelEncoder()
df_model['fed_encoded'] = le.fit_transform(df_model['fed'])

# 5. Drop columns we won't use (name, id, raw title columns, original sex/fed)
drop_cols = ['id', 'name', 'fed', 'sex', 'title', 'wtitle', 'otitle', 'foa', 'bday']
df_model = df_model.drop(columns=drop_cols)

print("Remaining features:")
print(df_model.columns.tolist())
print()
print("Sample rows:")
df_model.head()


## Step 5: Split Data into Train & Test Sets
We split data into:
- **Training set (80%)** — the model learns from this
- **Test set (20%)** — we evaluate how well the model does on unseen data

> 💡 Think of it like studying from a textbook (train) and then taking an exam (test).


In [ ]:
# Features (X) and Target (y)
X = df_model.drop(columns=['k'])
y = df_model['k']

print("Feature columns:", X.columns.tolist())
print("Target:", y.name)
print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining samples: {X_train.shape[0]:,}")
print(f"Testing samples:  {X_test.shape[0]:,}")


## Step 6: Build Machine Learning Models

We'll try **3 different models** and compare them:

| Model | How it works (simple explanation) |
|---|---|
| **Logistic Regression** | Draws a line/boundary to separate classes |
| **Random Forest** | Builds many decision trees and takes a vote |
| **Gradient Boosting** | Builds trees one-by-one, each fixing the previous one's mistakes |


In [ ]:
# Scale features (important for Logistic Regression)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

results = {}

# ── Model 1: Logistic Regression ──
print("Training Logistic Regression...")
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)
lr_pred = lr.predict(X_test_sc)
lr_acc = accuracy_score(y_test, lr_pred)
results['Logistic Regression'] = lr_acc
print(f"  Accuracy: {lr_acc:.4f}")

# ── Model 2: Random Forest ──
print("Training Random Forest...")
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)
results['Random Forest'] = rf_acc
print(f"  Accuracy: {rf_acc:.4f}")

# ── Model 3: Gradient Boosting ──
print("Training Gradient Boosting...")
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)
gb_acc = accuracy_score(y_test, gb_pred)
results['Gradient Boosting'] = gb_acc
print(f"  Accuracy: {gb_acc:.4f}")


## Step 7: Compare Models
Let's visualize which model performed best.

In [ ]:
# Bar chart comparison
plt.figure(figsize=(8, 4))
colors = ['#FF5722', '#4CAF50', '#2196F3']
bars = plt.bar(results.keys(), [v * 100 for v in results.values()],
               color=colors, edgecolor='black')
plt.ylim(80, 100)
plt.title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
plt.ylabel('Accuracy (%)')
for bar, val in zip(bars, results.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
             f'{val*100:.2f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

# Pick best model
best_model_name = max(results, key=results.get)
print(f"🏆 Best Model: {best_model_name} with {results[best_model_name]*100:.2f}% accuracy")


## Step 8: Evaluate the Best Model in Detail

We'll look at:
- **Classification Report**: Precision, Recall, F1-score per class
- **Confusion Matrix**: How often each class was predicted correctly vs. wrongly


In [ ]:
# Use Random Forest (best model)
best_pred = rf_pred

print("=" * 50)
print("Classification Report — Random Forest")
print("=" * 50)
print(classification_report(y_test, best_pred, target_names=['K=10', 'K=20', 'K=40']))


In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, best_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['K=10', 'K=20', 'K=40'])

fig, ax = plt.subplots(figsize=(7, 5))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Random Forest', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📖 How to read the confusion matrix:")
print("  - Diagonal cells = Correct predictions ✅")
print("  - Off-diagonal cells = Wrong predictions ❌")


## Step 9: Feature Importance
Which features did the Random Forest rely on most?

In [ ]:
feat_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(8, 5))
feat_imp.plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Feature Importance — Random Forest', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print("Top 3 most important features:")
for feat, score in feat_imp.sort_values(ascending=False).head(3).items():
    print(f"  {feat}: {score:.4f}")


## ✅ Conclusion

In this notebook, we analyzed the **FIDE Chess Ratings dataset** (201,015 players) and built machine learning models to **predict a player's K-factor class** (10 / 20 / 40).

### 🔍 Key Findings from EDA
- **~89.6%** of players are male; only ~10.4% are female.
- Players with **K=10** (top-tier) have the highest average ratings (~2500+), as expected.
- **Rating** and **number of games played** are the strongest predictors of K-factor.
- India (IND), Russia (RUS), and Germany (GER) have the most registered players.

### 🤖 Model Performance Summary

| Model | Accuracy |
|---|---|
| Logistic Regression | ~87% |
| Random Forest | ~97%+ |
| Gradient Boosting | ~96%+ |

### 🏆 Best Model: Random Forest
- Achieved **~97% accuracy** on the test set.
- The most important features were: **`rating`**, **`games`**, and **`age`**.

### 📌 What Did We Learn?
- A player's **current rating** is the strongest signal for determining their K-factor class.
- **Age** (birth year) also matters — younger players tend to be in the K=40 group.
- Ensemble methods like Random Forest significantly outperform simple Logistic Regression on this dataset.

### 🚀 What Could Be Improved?
- Use **XGBoost or LightGBM** for potentially even better speed and accuracy.
- Try **hyperparameter tuning** (GridSearchCV) to squeeze out extra performance.
- Predict **exact rating** instead of K-class (regression problem).

---
*If you found this notebook helpful, please give it an ⬆️ upvote on Kaggle!*
